# NYC short-term rental price exploration

This notebook explores the raw `sample.csv` artifact from the public Weights & Biases project `nyc_airbnb`. The goal is to decide which cleaning steps belong in the reusable `basic_cleaning` component.

Decisions recorded here:

- Keep nightly prices in the inclusive range `$10 <= price <= $350`. Stakeholders treat values outside that band as data-entry outliers, not market prices we want the model to learn.
- Convert `last_review` from text to a datetime so downstream steps can compute how long it has been since the last review.
- Do **not** impute or drop missing values in this notebook. The training pipeline imputes them inside the scikit-learn `Pipeline`, which means the same rules run at prediction time. Filling them here would hide that requirement and could train the model on a different treatment than production.


In [1]:
import wandb
import pandas as pd
import matplotlib.pyplot as plt

run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)
print(f"Loaded {local_path}")
print(f"Shape: {df.shape}")


/opt/homebrew/Caskroom/miniforge/base/envs/eda/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2422: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/homebrew/Caskroom/miniforge/base/envs/eda/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2422: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. Th

wandb: Currently logged in as: dharacoding (dharacoding-dhara-coding) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: creating run


wandb: Tracking run with wandb version 0.21.3


wandb: Run data is saved locally in /Users/rahulkumar/Documents/GitHub/UcityBuild_ml_pipeline_short_term_rental_Price/wandb/run-20261115_125231-ccpoirry
wandb: Run `wandb offline` to turn off syncing.


wandb: Syncing run lunar-dream-7


wandb: ⭐️ View project at https://wandb.ai/dharacoding-dhara-coding/nyc_airbnb


wandb: 🚀 View run at https://wandb.ai/dharacoding-dhara-coding/nyc_airbnb/runs/ccpoirry


Loaded ./artifacts/sample.csv:latest/sample1.csv
Shape: (20000, 16)


## Dataset structure

Check the column names, dtypes, and a few rows before changing anything. `last_review` arrives as text even though it is a date. Several columns are numeric counts or coordinates.


In [2]:
df.head()


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,9138664,Private Lg Room 15 min to Manhattan,47594947,Iris,Queens,Sunnyside,40.74271,-73.92493,Private room,74,2,6,2019-05-26,0.13,1,5
1,31444015,TIME SQUARE CHARMING ONE BED IN HELL'S KITCHEN...,8523790,Johlex,Manhattan,Hell's Kitchen,40.76682,-73.98878,Entire home/apt,170,3,0,NaN,NaN,1,188
2,8741020,Voted #1 Location Quintessential 1BR W Village...,45854238,John,Manhattan,West Village,40.73631,-74.00611,Entire home/apt,245,3,51,2018-09-19,1.12,1,0
3,34602077,Spacious 1 bedroom apartment 15min from Manhattan,261055465,Regan,Queens,Astoria,40.76424,-73.92351,Entire home/apt,125,3,1,2019-05-24,0.65,1,13
4,23203149,Big beautiful bedroom in huge Bushwick apartment,143460,Megan,Brooklyn,Bushwick,40.69839,-73.92044,Private room,65,2,8,2019-06-23,0.52,2,8


In [3]:
df.dtypes


id                                  int64
name                               object
host_id                             int64
host_name                          object
neighbourhood_group                object
neighbourhood                      object
latitude                          float64
longitude                         float64
room_type                          object
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
dtype: object

In [4]:
df.describe(include="all").T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,20000.0,NaN,NaN,NaN,18923800.78335,11012232.415483,2539.0,9393540.5,19521168.5,29129358.75,36485609.0
name,19993,19768,Brooklyn Apartment,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN
host_id,20000.0,NaN,NaN,NaN,67460344.0675,78579364.799929,2571.0,7853718.25,31114309.5,106842560.0,274273284.0
host_name,19992,6517,David,170,NaN,NaN,NaN,NaN,NaN,NaN,NaN
neighbourhood_group,20000,5,Manhattan,8774,NaN,NaN,NaN,NaN,NaN,NaN,NaN
neighbourhood,20000,217,Williamsburg,1580,NaN,NaN,NaN,NaN,NaN,NaN,NaN
latitude,20000.0,NaN,NaN,NaN,40.728455,0.054755,40.50873,40.68942,40.72273,40.76299,40.91306
longitude,20000.0,NaN,NaN,NaN,-73.952125,0.046559,-74.23914,-73.98303,-73.95564,-73.93638,-73.71795
room_type,20000,3,Entire home/apt,10384,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,20000.0,NaN,NaN,NaN,153.26905,243.325609,0.0,69.0,105.0,175.0,10000.0


## Missing values and duplicates

Missingness is concentrated in review fields (`last_review`, `reviews_per_month`) and, less often, in listing or host names. A listing with no reviews is a real state, not a value we should invent during exploration.

Imputation is deferred to the inference pipeline on purpose. `train_random_forest` builds a scikit-learn pipeline that:

- fills `neighbourhood_group` with the most frequent category before one-hot encoding;
- fills numeric counts and coordinates with 0;
- fills a missing `last_review` with an old sentinel date so "never reviewed" becomes a large day-delta;
- fills a missing `name` with an empty string before TF-IDF.

Those transformers are saved with the model, so a new listing with the same gaps is handled the same way. Imputing or dropping rows in this notebook would make the training table look cleaner than the data the model will see in production.


In [5]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing.to_string())
print()
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate ids: {df['id'].duplicated().sum()}")


Columns with missing values:
last_review          4123
reviews_per_month    4123
host_name               8
name                    7

Duplicate rows: 0
Duplicate ids: 0


## Price distribution

The raw price column includes zeros and extreme nightly rates. After checking the distribution with stakeholders, the accepted market range for this pipeline is **$10 to $350 inclusive**. Rows outside that range are removed in cleaning. They are not winsorized, because a $10,000 "price" is not a plausible observation we want the forest to treat as a high-but-real rent.


In [6]:
print(df["price"].describe())
print()
print(f"Prices below $10: {(df['price'] < 10).sum()}")
print(f"Prices above $350: {(df['price'] > 350).sum()}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["price"], bins=50, color="#4c78a8")
axes[0].set_title("Raw price")
axes[0].set_xlabel("price")
clipped = df.loc[df["price"].between(10, 350), "price"]
axes[1].hist(clipped, bins=40, color="#f58518")
axes[1].set_title("Price inside $10 to $350")
axes[1].set_xlabel("price")
fig.tight_layout()
plt.show()


count    20000.000000
mean       153.269050
std        243.325609
min          0.000000
25%         69.000000
50%        105.000000
75%        175.000000
max      10000.000000
Name: price, dtype: float64

Prices below $10: 5
Prices above $350: 994


/var/folders/7z/5qb9wcyx7z9b5_ff36bxqt8c0000gn/T/ipykernel_9672/2195852262.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Geography

Listings should sit in New York City. Latitude and longitude are inspected here so a later release can reject points outside the city bounding box (longitude -74.25 to -73.50, latitude 40.5 to 41.2). This first exploration does not drop geographic outliers; that filter is the 1.0.1 cleaning change, after the boundary test has a chance to catch a bad sample.


In [7]:
print(df[["latitude", "longitude"]].describe())
in_nyc = df["longitude"].between(-74.25, -73.50) & df["latitude"].between(40.5, 41.2)
print(f"Rows outside the NYC bounding box: {(~in_nyc).sum()}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(df["longitude"], df["latitude"], s=4, alpha=0.25, c="#4c78a8")
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title("Listing coordinates")
plt.show()


           latitude     longitude
count  20000.000000  20000.000000
mean      40.728455    -73.952125
std        0.054755      0.046559
min       40.508730    -74.239140
25%       40.689420    -73.983030
50%       40.722730    -73.955640
75%       40.762990    -73.936380
max       40.913060    -73.717950
Rows outside the NYC bounding box: 0


/var/folders/7z/5qb9wcyx7z9b5_ff36bxqt8c0000gn/T/ipykernel_9672/2177123152.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Categorical features

`neighbourhood_group` and `room_type` are the categories the model encodes. The five boroughs are the expected groups. `room_type` is ordinal in the training pipeline (`Entire home/apt` > `Private room` > `Shared room`) because the room type is required on the booking sites and is not imputed. `neighbourhood_group` is not ordinal: it is imputed with the most frequent value and one-hot encoded.


In [8]:
print("neighbourhood_group")
print(df["neighbourhood_group"].value_counts(dropna=False).to_string())
print()
print("room_type")
print(df["room_type"].value_counts(dropna=False).to_string())
print()
print(f"Distinct neighbourhoods: {df['neighbourhood'].nunique(dropna=True)}")


neighbourhood_group
neighbourhood_group
Manhattan        8774
Brooklyn         8265
Queens           2355
Bronx             441
Staten Island     165

room_type
room_type
Entire home/apt    10384
Private room        9172
Shared room          444

Distinct neighbourhoods: 217


## Profile report

`ydata-profiling` summarizes types, missingness, and the price outliers already visible above. A second profile is built after cleaning to confirm the price filter and the `last_review` conversion.


In [9]:
from ydata_profiling import ProfileReport

raw_profile = ProfileReport(df, title="NYC Airbnb raw sample", minimal=True)
raw_profile.to_file("eda_profile_raw.html")
description = raw_profile.get_description()
n_vars = len(getattr(description, "variables", {}) or {})
print(f"Raw profile variables: {n_vars}")
print(f"Raw profile alerts: {len(getattr(description, 'alerts', []) or [])}")
print("Wrote eda_profile_raw.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 582.26it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Raw profile variables: 16
Raw profile alerts: 6
Wrote eda_profile_raw.html


## Cleaning applied in this notebook

The same two operations are reimplemented in `src/basic_cleaning`:

1. Keep rows whose price is between $10 and $350 inclusive.
2. Parse `last_review` as a datetime.

Missing values are retained. There is no `fillna` and no row drop based on nulls.


In [10]:
min_price = 10
max_price = 350
idx = df["price"].between(min_price, max_price)
print(f"Dropping {(~idx).sum()} rows outside [{min_price}, {max_price}]")
df = df[idx].copy()
df["last_review"] = pd.to_datetime(df["last_review"])
print(f"Clean shape: {df.shape}")
print(f"last_review dtype: {df['last_review'].dtype}")


Dropping 999 rows outside [10, 350]
Clean shape: (19001, 16)
last_review dtype: datetime64[ns]


## Recheck after cleaning

Price now stays inside the agreed range. `last_review` is a datetime, and the review columns are still allowed to be missing. That matches the checklist in `notes.md`.


In [11]:
df.info()
print()
print(df["price"].describe())
print()
print("Missing values after cleaning:")
print(df.isna().sum()[df.isna().sum() > 0].to_string())
assert df["price"].between(min_price, max_price).all()
assert str(df["last_review"].dtype).startswith("datetime64")


<class 'pandas.core.frame.DataFrame'>
Index: 19001 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              19001 non-null  int64         
 1   name                            18994 non-null  object        
 2   host_id                         19001 non-null  int64         
 3   host_name                       18993 non-null  object        
 4   neighbourhood_group             19001 non-null  object        
 5   neighbourhood                   19001 non-null  object        
 6   latitude                        19001 non-null  float64       
 7   longitude                       19001 non-null  float64       
 8   room_type                       19001 non-null  object        
 9   price                           19001 non-null  int64         
 10  minimum_nights                  19001 non-null  int64         
 11  number_

In [12]:
clean_profile = ProfileReport(df, title="NYC Airbnb cleaned sample", minimal=True)
clean_profile.to_file("eda_profile_clean.html")
description = clean_profile.get_description()
n_vars = len(getattr(description, "variables", {}) or {})
print(f"Clean profile variables: {n_vars}")
print("Wrote eda_profile_clean.html")
print(f"Rows remaining: {len(df)}")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 569.24it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Clean profile variables: 16
Wrote eda_profile_clean.html
Rows remaining: 19001


## Finish the W&B run

`save_code=True` stores this notebook with the EDA run. The run is closed here so the artifact use is recorded.


In [13]:
run.finish()


wandb: updating run metadata


wandb: uploading output.log; uploading wandb-summary.json


wandb: uploading summary, console lines 0-101


wandb:                                                                                


wandb: 🚀 View run lunar-dream-7 at: https://wandb.ai/dharacoding-dhara-coding/nyc_airbnb/runs/ccpoirry
wandb: ⭐️ View project at: https://wandb.ai/dharacoding-dhara-coding/nyc_airbnb
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)


wandb: Find logs at: ./wandb/run-20261115_125231-ccpoirry/logs
